# 02 · Línea base

Evalúa el modelo **sin LoRA** (zero-shot con el mismo prompt) sobre el test set. Es el punto de comparación de todos los experimentos.

In [ ]:
# === Setup ===
# En Google Colab: clona la rama del proyecto, instala el paquete y monta Drive.
# Fuera de Colab (local, desde la raíz del repo) esta celda no hace nada.
import os, sys
EN_COLAB = "google.colab" in sys.modules
RAMA = "splits-con-manual"  # cambiar a "main" cuando la rama se fusione
if EN_COLAB:
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone -q -b {RAMA} https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    # Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16); no lo usamos.
    !pip uninstall -q -y torchao
    # Colab ya trae las dependencias: esto solo verifica versiones mínimas e instala `triaje`.
    !pip install -q -r requirements.txt
    !pip install -q -e .
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
if EN_COLAB:
    # Resultados y adaptadores a Drive: sobreviven si Colab se desconecta, y la
    # suite reanuda leyendo el resultados.csv de Drive.
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir = config.drive_dir / "adaptadores"
set_seed(config.seed)
print("modelo:", config.model_name, "| resultados en:", config.results_dir)

## Cargar modelo base y evaluar en test

In [ ]:
from triaje.modelo import cargar_modelo_y_tokenizer
from triaje.data import cargar_split
from triaje.evaluate import evaluar

test = cargar_split(config.data_dir / "splits", "test")
model, tok = cargar_modelo_y_tokenizer(config, para_entrenar=False)
metricas = evaluar(model, tok, test, config,
                   guardar_predicciones_en=str(config.results_dir / "pred_base.jsonl"))
print("Esquema OK (post):", metricas.esquema_ok_post)
print("Exact match categóricos:", metricas.exact_match_categoricos)
print("Macro-F1:", metricas.macro_f1)
print("Tasa de sub-triaje:", metricas.tasa_subtriaje)